In [2]:
import cmath
import math
import numpy as np

# ==========================================
# Hardware Quantization Configuration
# ==========================================
D_BITS = 18; D_FRAC = 17
T_BITS = 20; T_FRAC = 17

def to_hex(val, bits, frac_bits):
    # Noise gate to protect math.floor from floating-point anomalies
    if abs(val) < 1e-10:
        val = 0.0
        
    scaled = math.floor(val * (1 << frac_bits))
    max_val = (1 << (bits - 1)) - 1
    min_val = -(1 << (bits - 1))
    
    if scaled > max_val: scaled = max_val
    elif scaled < min_val: scaled = min_val
        
    mask = (1 << bits) - 1
    hex_chars = (bits + 3) // 4
    return f"0x{(scaled & mask):0{hex_chars}X}"

def hex_cpx(c_val, bits, frac_bits):
    return f"{to_hex(c_val.real, bits, frac_bits)} + j {to_hex(c_val.imag, bits, frac_bits)}"

# ==========================================
# FFT Math Core
# ==========================================
def w(n, k):
    return cmath.exp(-2j * cmath.pi * k / n)

def radix4_butterfly(a, b, c, d, w1, w2, w3):
    s0 = a + c; s1 = a - c; s2 = b + d; s3 = b - d
    A_pre = s0 + s2; B_pre = s1 - 1j * s3; C_pre = s0 - s2; D_pre = s1 + 1j * s3
    return A_pre, B_pre * w1, C_pre * w2, D_pre * w3

def radix2_butterfly(a, b):
    return a + b, a - b

# ==========================================
# MDC Pipeline Simulation with Clock Logging
# ==========================================
def simulate_mdc_pipeline(input_data):
    # ---------------------------------------------------------
    # STAGE 1: Radix-4 (Scale by /4)
    # ---------------------------------------------------------
    stage1_out = [0j] * 128
    print("\n" + "="*95)
    print(" STAGE 1 (Radix-4) - Generating 32 Clock Cycles")
    print("="*95)
    for i in range(32):
        a, b, c, d = input_data[i], input_data[i+32], input_data[i+64], input_data[i+96]
        w1, w2, w3 = w(128, i), w(128, 2*i), w(128, 3*i)
        
        A, B, C, D = radix4_butterfly(a, b, c, d, w1, w2, w3)
        
        # Apply Hardware Datapath Scaling immediately
        A_hw, B_hw, C_hw, D_hw = A/4, B/4, C/4, D/4
        stage1_out[i], stage1_out[i+32], stage1_out[i+64], stage1_out[i+96] = A_hw, B_hw, C_hw, D_hw

        print(f"[Clk {i:02d}] In : a={hex_cpx(a, D_BITS, D_FRAC)}, b={hex_cpx(b, D_BITS, D_FRAC)}, c={hex_cpx(c, D_BITS, D_FRAC)}, d={hex_cpx(d, D_BITS, D_FRAC)}")
        print(f"          Out: A={hex_cpx(A_hw, D_BITS, D_FRAC)}, B={hex_cpx(B_hw, D_BITS, D_FRAC)}, C={hex_cpx(C_hw, D_BITS, D_FRAC)}, D={hex_cpx(D_hw, D_BITS, D_FRAC)}")

    # ---------------------------------------------------------
    # STAGE 2: Radix-4 (Scale by /4)
    # ---------------------------------------------------------
    stage2_out = [0j] * 128
    print("\n" + "="*95)
    print(" STAGE 2 (Radix-4) - Generating 32 Clock Cycles")
    print("="*95)
    clk = 0
    for block in range(4): 
        offset = block * 32
        for i in range(8):
            a, b, c, d = stage1_out[offset+i], stage1_out[offset+i+8], stage1_out[offset+i+16], stage1_out[offset+i+24]
            w1, w2, w3 = w(32, i), w(32, 2*i), w(32, 3*i)
            
            A, B, C, D = radix4_butterfly(a, b, c, d, w1, w2, w3)
            
            A_hw, B_hw, C_hw, D_hw = A/4, B/4, C/4, D/4
            stage2_out[offset+i], stage2_out[offset+i+8], stage2_out[offset+i+16], stage2_out[offset+i+24] = A_hw, B_hw, C_hw, D_hw

            print(f"[Clk {clk:02d}] In : a={hex_cpx(a, D_BITS, D_FRAC)}, b={hex_cpx(b, D_BITS, D_FRAC)}, c={hex_cpx(c, D_BITS, D_FRAC)}, d={hex_cpx(d, D_BITS, D_FRAC)}")
            print(f"          Out: A={hex_cpx(A_hw, D_BITS, D_FRAC)}, B={hex_cpx(B_hw, D_BITS, D_FRAC)}, C={hex_cpx(C_hw, D_BITS, D_FRAC)}, D={hex_cpx(D_hw, D_BITS, D_FRAC)}")
            clk += 1

    # ---------------------------------------------------------
    # STAGE 3: Radix-4 (Scale by /4)
    # ---------------------------------------------------------
    stage3_out = [0j] * 128
    print("\n" + "="*95)
    print(" STAGE 3 (Radix-4) - Generating 32 Clock Cycles")
    print("="*95)
    clk = 0
    for block in range(16): 
        offset = block * 8
        for i in range(2):
            a, b, c, d = stage2_out[offset+i], stage2_out[offset+i+2], stage2_out[offset+i+4], stage2_out[offset+i+6]
            w1, w2, w3 = w(8, i), w(8, 2*i), w(8, 3*i)
            
            A, B, C, D = radix4_butterfly(a, b, c, d, w1, w2, w3)
            
            A_hw, B_hw, C_hw, D_hw = A/4, B/4, C/4, D/4
            stage3_out[offset+i], stage3_out[offset+i+2], stage3_out[offset+i+4], stage3_out[offset+i+6] = A_hw, B_hw, C_hw, D_hw

            print(f"[Clk {clk:02d}] In : a={hex_cpx(a, D_BITS, D_FRAC)}, b={hex_cpx(b, D_BITS, D_FRAC)}, c={hex_cpx(c, D_BITS, D_FRAC)}, d={hex_cpx(d, D_BITS, D_FRAC)}")
            print(f"          Out: A={hex_cpx(A_hw, D_BITS, D_FRAC)}, B={hex_cpx(B_hw, D_BITS, D_FRAC)}, C={hex_cpx(C_hw, D_BITS, D_FRAC)}, D={hex_cpx(D_hw, D_BITS, D_FRAC)}")
            clk += 1

    # ---------------------------------------------------------
    # STAGE 4: Radix-2 Finisher (Scale by /2)
    # ---------------------------------------------------------
    final_output = [0j] * 128
    print("\n" + "="*95)
    print(" STAGE 4 (Radix-2 Finisher) - Generating 64 Clock Cycles")
    print("="*95)
    clk = 0
    for block in range(64):
        offset = block * 2
        a, b = stage3_out[offset], stage3_out[offset + 1]
        
        A, B = radix2_butterfly(a, b)
        
        A_hw, B_hw = A/2, B/2
        final_output[offset], final_output[offset + 1] = A_hw, B_hw
        
        print(f"[Clk {clk:02d}] In : a={hex_cpx(a, D_BITS, D_FRAC)}, b={hex_cpx(b, D_BITS, D_FRAC)}")
        print(f"          Out: A={hex_cpx(A_hw, D_BITS, D_FRAC)}, B={hex_cpx(B_hw, D_BITS, D_FRAC)}")
        clk += 1

    return final_output

def digit_reverse_128(index):
    stage1 = (index >> 5) & 0x3 
    stage2 = (index >> 3) & 0x3 
    stage3 = (index >> 1) & 0x3 
    stage4 = index & 0x1        
    return (stage4 << 6) | (stage3 << 4) | (stage2 << 2) | stage1

# ==========================================
# Execution: Random Generation & Export
# ==========================================
if __name__ == "__main__":
    # 1. Generate a purely random input signal
    np.random.seed(42) # Seeded for reproducible results
    random_signal = np.random.uniform(-0.99, 0.99, 128) + 0j
    
    # 2. Write to input_signal.mem for Verilog Testbench
    with open("input_signal.mem", "w") as f:
        for val in random_signal:
            hex_str = to_hex(val.real, D_BITS, D_FRAC).replace("0x", "")
            f.write(f"{hex_str}\n")
    print("Generated 'input_signal.mem' with 128 random samples.")

    # 3. Run Hardware Simulation (Prints every clock cycle)
    hw_result_raw = simulate_mdc_pipeline(list(random_signal))
    
    # 4. Re-order results to match standard FFT output
    hw_result = [0j] * 128
    for j in range(128):
        hw_result[digit_reverse_128(j)] = hw_result_raw[j]
        
    # 5. Verify against pure math
    ideal_result = np.fft.fft(random_signal)
    
    # Hardware mathematically scaled down by 128 (4 * 4 * 4 * 2). 
    # To find the absolute error, we must scale the hardware result back up.
    hw_result_unscaled = np.array(hw_result) * 128.0
    error = np.max(np.abs(hw_result_unscaled - ideal_result))
    
    signal_power = np.sum(np.abs(ideal_result)**2)
    noise_power = np.sum(np.abs(hw_result_unscaled - ideal_result)**2)
    sqnr = 10 * np.log10(signal_power / noise_power) if noise_power > 0 else float('inf')
    
    print("\n===================================================")
    print("             DSP Verification Metrics              ")
    print("===================================================")
    print(f"Mathematical Error : {error:.4e}")
    print(f"Pipeline SQNR      : {sqnr:.2f} dB")
    print("===================================================")

Generated 'input_signal.mem' with 128 random samples.

 STAGE 1 (Radix-4) - Generating 32 Clock Cycles
[Clk 00] In : a=0x380D0 + j 0x00000, b=0x24711 + j 0x00000, c=0x321EB + j 0x00000, d=0x0170B + j 0x00000
          Out: A=0x34036 + j 0x00000, B=0x017B9 + j 0x073FE, C=0x01127 + j 0x00000, D=0x017B9 + j 0x38C01
[Clk 01] In : a=0x1C8EA + j 0x00000, b=0x1C70F + j 0x00000, c=0x02B48 + j 0x00000, d=0x3B68B + j 0x00000
          Out: A=0x0DC73 + j 0x00000, B=0x060CC + j 0x376F4, C=0x01D81 + j 0x3FD18, D=0x079AD + j 0x07386
[Clk 02] In : a=0x0EB2F + j 0x00000, b=0x1D80A + j 0x00000, c=0x293FB + j 0x00000, d=0x21EE3 + j 0x00000
          Out: A=0x3DD86 + j 0x00000, B=0x07DB9 + j 0x3042D, C=0x3E2A2 + j 0x005D7, D=0x0D485 + j 0x0B88A
[Clk 03] In : a=0x06404 + j 0x00000, b=0x138A4 + j 0x00000, c=0x1325A + j 0x00000, d=0x2727E + j 0x00000
          Out: A=0x05060 + j 0x00000, B=0x3B2EC + j 0x357F4, C=0x07585 + j 0x3DC59, D=0x01D46 + j 0x0B68B
[Clk 04] In : a=0x2A349 + j 0x00000, b=0x339EC + j 0x